# <span style="color:#0b486b">  FIT5221 - Assignment 3: Task 1, 2 </span>


## <span style="color:#0b486b">  Student Information</span>
***
Fullname: **[Yin Zhang]**  <br/>
Student ID: **[[Student ID removed]]**    <br/>
***

In [ ]:
! pip install torchinfo -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install huggingface_hub --index-url https://pypi.tuna.tsinghua.edu.cn/simple
! pip install transformers --no-deps --no-cache-dir --index-url https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
# import library
import os
import torch
from torchvision.datasets import VOCSegmentation
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
from torchinfo import summary
from torchvision.models import efficientnet_b0
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms
import copy

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("Pytorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
set_seed(42) # For reproduciblity purpose, please do not modify this.

import sys
from pathlib import Path

def _ensure_a3_dataset_mirrors_on_path():
    """Ensure a3_dataset_mirrors.py on sys.path regardless of notebook cwd."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and hasattr(ip, "startup_dir"):
            candidates.insert(0, Path(ip.startup_dir))
    except Exception:
        pass
    for d in candidates:
        if (d / "a3_dataset_mirrors.py").is_file():
            s = str(d.resolve())
            if s not in sys.path:
                sys.path.insert(0, s)
            return d
    raise FileNotFoundError(
        "Cannot find a3_dataset_mirrors.py. Open the notebook from the FIT5221_Assignment_3 project root, "
        "or place a3_dataset_mirrors.py next to the notebook."
    )

_ensure_a3_dataset_mirrors_on_path()

import importlib
import a3_dataset_mirrors as _mirrors
importlib.reload(_mirrors)  # Avoid stale cached module in Jupyter
from a3_dataset_mirrors import download_voc2012, load_voc_segmentation

_mirrors.setup_china_mirrors()  # Tsinghua PyPI + HF mirror
_mirrors.print_mirror_plan("all", root=".")  # Download plan and checkpoint skip status


# Helper functions and dataset setup

**Mirror setup (China-friendly):**
- Python deps (pip/uv): [Tsinghua TUNA PyPI mirror](https://mirrors.tuna.tsinghua.edu.cn/help/pypi/).
- HuggingFace: `HF_ENDPOINT=https://hf-mirror.com`.
- Pascal VOC 2012: download via Baidu BOS (`dataset.bj.bcebos.com`); see `a3_dataset_mirrors.py`.

## 1. Download dataset
Please refer to [this function](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.VOCSegmentation.html) from TorchVision to download the Pascal VOC Segmentation Dataset. You can change the input of provided code to match with your requirement.

Because the Pascal VOC Segmentation Dataset 2012 only provide a `train` set and a `val` set. So that you are required to train on `train` set only and then test the model on `val` set

**Note:** there is a void class with index 255 in the dataset. If you are using PyTorch’s CrossEntropyLoss, please ignore void class by setting `ignore_index=255`. Refer to this post for further information: https://discuss.pytorch.org/t/having-trouble-with-voc-2012-segmentation-with-the-void-255-label/46486


In [ ]:
from a3_dataset_mirrors import download_voc2012, load_voc_segmentation

voc_dir = './data'
os.makedirs(voc_dir, exist_ok=True)
# Mirror: Baidu BOS (download_voc2012 prints plan)
download_voc2012(voc_dir)
train_dataset = load_voc_segmentation(voc_dir, image_set='train')
val_dataset = load_voc_segmentation(voc_dir, image_set='val')


VOC_CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle",
               "bus", "car",  "cat",  "chair", "cow",  "diningtable", "dog", "horse",
               "motorbike", "person","potted plant", "sheep", "sofa","train", "tv/monitor"]

VOC_COLORMAP = [
    [0, 0, 0],
    [128, 0, 0],
    [0, 128, 0],
    [128, 128, 0],
    [0, 0, 128],
    [128, 0, 128],
    [0, 128, 128],
    [128, 128, 128],
    [64, 0, 0],
    [192, 0, 0],
    [64, 128, 0],
    [192, 128, 0],
    [64, 0, 128],
    [192, 0, 128],
    [64, 128, 128],
    [192, 128, 128],
    [0, 64, 0],
    [128, 64, 0],
    [0, 192, 0],
    [128, 192, 0],
    [0, 64, 128],
]


## 2. Helper function
You are required to use this helper function to calculate the mean IoU score

In [ ]:
def accumulate_segmentation_counts(preds, masks, num_classes=21, ignore_index=255):
    """
    Count TP, FP, and FN for each class in one batch.

    Void pixels with 'ignore_index' are removed before counting.
    The returned counts can be added across batches and used to compute the final metrics.
    """

    preds = preds.view(-1)
    masks = masks.view(-1)

    # remove void pixels from BOTH preds and masks
    if ignore_index is not None:
        valid = masks != ignore_index
        preds = preds[valid]
        masks = masks[valid]

    # Initialize counters for this batch
    tp = torch.zeros(num_classes, dtype=torch.float64)
    fp = torch.zeros(num_classes, dtype=torch.float64)
    fn = torch.zeros(num_classes, dtype=torch.float64)
    
    # skip background class (cls=0).
    for cls in range(1, num_classes):
        pred_inds = (preds == cls)
        target_inds = (masks == cls)

        tp[cls] = (pred_inds & target_inds).sum().item()
        fp[cls] = (pred_inds & ~target_inds).sum().item()
        fn[cls] = (~pred_inds & target_inds).sum().item()

    return tp, fp, fn


def calculate_segmentation_metrics(global_tp, global_fp, global_fn, num_classes=21):
    """
    Compute class-averaged foreground segmentation metrics from accumulated/global TP, FP, and FN counts.
    The background class (cls=0) is excluded from the calculation.
    """
    
    eps = 1e-6
    precision_list = []
    recall_list = []
    iou_list = []
    dice_list = []

    # skip background class (cls=0).
    for cls in range(1, num_classes):
        
        tp = global_tp[cls]
        fp = global_fp[cls]
        fn = global_fn[cls]

        # Skip classes with no ground truth and no predictions to avoid NaN
        if tp == 0 and fp == 0 and fn == 0:
            continue

        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        iou = tp / (tp + fp + fn + eps)
        dice = (2 * tp) / (2 * tp + fp + fn + eps)

        precision_list.append(precision.item())
        recall_list.append(recall.item())
        iou_list.append(iou.item())
        dice_list.append(dice.item())

    return {
        "precision": np.mean(precision_list) if precision_list else 0.0,
        "recall": np.mean(recall_list) if recall_list else 0.0,
        "iou": np.mean(iou_list) if iou_list else 0.0,
        "dice": np.mean(dice_list) if dice_list else 0.0,
    }

# Task 1: Build a baseline Fully Convolutional Network (FCN) model for semantic segmentation (5 marks)

In [ ]:
# EfficientNet-B0 backbone (assignment allows custom loading)
# Default: torchvision only (no GitHub); weights cached under ~/.cache/torch.
# For NVIDIA torch.hub: os.environ["USE_NVIDIA_TORCH_HUB"] = "1" before running
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0


def load_efficientnet_b0_backbone():
    """Load ImageNet-pretrained EfficientNet-B0 (torchvision by default)."""
    if os.environ.get("USE_NVIDIA_TORCH_HUB") == "1":
        try:
            return torch.hub.load(
                "NVIDIA/DeepLearningExamples:torchhub",
                "nvidia_efficientnet_b0",
                pretrained=True,
                trust_repo=True,
            )
        except Exception as e:
            print("NVIDIA torchhub failed, fallback to torchvision:", e)
    return efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)


backbone_efficientnet = load_efficientnet_b0_backbone()
print(type(backbone_efficientnet))
print(hasattr(backbone_efficientnet, "features"))

In [ ]:
# ===== Task 1: Baseline FCN =====

from pathlib import Path
from tqdm import tqdm

# Make sure dataset helper functions are imported
from a3_dataset_mirrors import download_voc2012

NUM_CLASSES = 21
IGNORE_INDEX = 255
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
CHECKPOINT_BASELINE = "checkpoints/baseline_fcn.pth"

# Hyperparameters for report
OPTIMIZER_NAME = "Adam"
LOSS_NAME = "CrossEntropyLoss(ignore_index=255)"


def get_efficientnet_features(backbone):
    if hasattr(backbone, "features"):
        return backbone.features
    raise AttributeError("Backbone must expose `.features`.")


class BaselineFCN(nn.Module):
    def __init__(self, backbone, num_classes=NUM_CLASSES):
        super().__init__()

        self.backbone = get_efficientnet_features(backbone)

        self.score = nn.Conv2d(
            in_channels=1280,
            out_channels=num_classes,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True
        )

        self.upsample = nn.ConvTranspose2d(
            in_channels=num_classes,
            out_channels=num_classes,
            kernel_size=64,
            stride=32,
            padding=16,
            bias=False
        )

    def forward(self, x):
        x = self.backbone(x)
        x = F.relu(self.score(x))
        x = self.upsample(x)
        return x


class VOCSegDataset(torch.utils.data.Dataset):
    def __init__(self, voc_dataset, transform_img, transform_mask):
        self.ds = voc_dataset
        self.transform_img = transform_img
        self.transform_mask = transform_mask

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        image, mask = self.ds[idx]
        image = self.transform_img(image)
        mask = self.transform_mask(mask).squeeze(0).long()
        return image, mask


def build_voc_loaders(batch_size=BATCH_SIZE):
    img_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    mask_transform = transforms.Compose([
        transforms.Resize(
            (IMG_SIZE, IMG_SIZE),
            interpolation=transforms.InterpolationMode.NEAREST
        ),
        transforms.PILToTensor(),
    ])

    print("Checking VOC dataset...")
    download_voc2012(voc_dir)

    train_raw = VOCSegmentation(
        root=voc_dir,
        year="2012",
        image_set="train",
        download=False
    )

    val_raw = VOCSegmentation(
        root=voc_dir,
        year="2012",
        image_set="val",
        download=False
    )

    train_ds = VOCSegDataset(train_raw, img_transform, mask_transform)
    val_ds = VOCSegDataset(val_raw, img_transform, mask_transform)

    # Important for Windows + Jupyter:
    # num_workers=0 avoids DataLoader hanging at 0/183.
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    print("DataLoader ready: num_workers=0, pin_memory=False")
    return train_loader, val_loader


def run_segmentation_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train(train)

    total_loss = 0.0
    correct = 0
    total = 0

    tp = torch.zeros(NUM_CLASSES, dtype=torch.float64)
    fp = torch.zeros(NUM_CLASSES, dtype=torch.float64)
    fn = torch.zeros(NUM_CLASSES, dtype=torch.float64)

    desc = "train" if train else "val"

    for images, masks in tqdm(loader, desc=desc, leave=False):
        images = images.to(device)
        masks = masks.to(device)

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        valid = masks != IGNORE_INDEX

        correct += (preds[valid] == masks[valid]).sum().item()
        total += valid.sum().item()

        btp, bfp, bfn = accumulate_segmentation_counts(
            preds.cpu(),
            masks.cpu(),
            NUM_CLASSES,
            IGNORE_INDEX
        )

        tp += btp
        fp += bfp
        fn += bfn

    metrics = calculate_segmentation_metrics(tp, fp, fn, NUM_CLASSES)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": correct / max(total, 1),
        "mean_iou": metrics["iou"],
        "metrics": metrics,
    }


def train_baseline_fcn(model):
    os.makedirs("checkpoints", exist_ok=True)

    train_loader, val_loader = build_voc_loaders(batch_size=BATCH_SIZE)

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_miou": [],
        "val_miou": [],
    }

    best_miou = -1.0
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

        train_stats = run_segmentation_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            train=True
        )

        val_stats = run_segmentation_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
            train=False
        )

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc"].append(train_stats["accuracy"])
        history["val_acc"].append(val_stats["accuracy"])
        history["train_miou"].append(train_stats["mean_iou"])
        history["val_miou"].append(val_stats["mean_iou"])

        print(
            f"train: loss={train_stats['loss']:.4f}, "
            f"acc={train_stats['accuracy']:.4f}, "
            f"mIoU={train_stats['mean_iou']:.4f}"
        )

        print(
            f"val:   loss={val_stats['loss']:.4f}, "
            f"acc={val_stats['accuracy']:.4f}, "
            f"mIoU={val_stats['mean_iou']:.4f}"
        )

        if val_stats["mean_iou"] > best_miou:
            best_miou = val_stats["mean_iou"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    torch.save(
        {
            "model_state_dict": best_state,
            "best_val_miou": best_miou,
            "history": history,
        },
        CHECKPOINT_BASELINE
    )

    print(f"\nSaved checkpoint: {CHECKPOINT_BASELINE}")
    print(f"Best validation mIoU: {best_miou:.4f}")

    return history


def plot_training_history(history, title_prefix="Baseline FCN"):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title(f"{title_prefix} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["val_acc"], label="val")
    axes[1].set_title(f"{title_prefix} - Pixel Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(epochs, history["train_miou"], label="train")
    axes[2].plot(epochs, history["val_miou"], label="val")
    axes[2].set_title(f"{title_prefix} - Mean IoU")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


def mask_to_color(mask):
    mask = mask.cpu().numpy().astype(np.int32)

    h, w = mask.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)

    for cls_id, color_rgb in enumerate(VOC_COLORMAP):
        color[mask == cls_id] = color_rgb

    return color


def visualize_predictions(model, dataset, num_samples=10, title="Validation samples"):
    model.eval()

    indices = random.sample(range(len(dataset)), num_samples)

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    with torch.no_grad():
        for row, idx in enumerate(indices):
            image, mask = dataset[idx]

            pred = model(image.unsqueeze(0).to(device))
            pred = pred.argmax(dim=1).squeeze(0)

            img_vis = (image * std + mean).permute(1, 2, 0).numpy()
            img_vis = np.clip(img_vis, 0, 1)

            axes[row, 0].imshow(img_vis)
            axes[row, 0].set_title("Input")

            axes[row, 1].imshow(mask_to_color(mask))
            axes[row, 1].set_title("Ground Truth")

            axes[row, 2].imshow(mask_to_color(pred))
            axes[row, 2].set_title("Prediction")

            for col in range(3):
                axes[row, col].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def evaluate_checkpoint(model, checkpoint_path, split="val"):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)

    _, val_loader = build_voc_loaders(batch_size=BATCH_SIZE)

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    stats = run_segmentation_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        train=False
    )

    print(f"Loaded checkpoint: {checkpoint_path}")
    print(f"Saved best val mIoU: {ckpt.get('best_val_miou', 'N/A')}")
    print(
        f"Current eval ({split}): "
        f"loss={stats['loss']:.4f}, "
        f"acc={stats['accuracy']:.4f}, "
        f"mIoU={stats['mean_iou']:.4f}"
    )

    return stats


# Build and summarize the baseline model only.
baseline_model = BaselineFCN(backbone_efficientnet).to(device)

print(summary(baseline_model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), device=device))
print("Total params:", sum(p.numel() for p in baseline_model.parameters()))

In [ ]:
print("Before building loaders")

train_loader, val_loader = build_voc_loaders(batch_size=1)

print("Loaders built")
print("train size:", len(train_loader.dataset))
print("val size:", len(val_loader.dataset))

print("Before dataset[0]")
image0, mask0 = train_loader.dataset[0]
print("dataset[0] ok:", image0.shape, mask0.shape)

print("Before next(iter(train_loader))")
images, masks = next(iter(train_loader))

print("next(iter) ok")
print("images:", images.shape)
print("masks:", masks.shape)
print("mask min/max:", masks.min().item(), masks.max().item())

In [ ]:
from pathlib import Path

print(Path(CHECKPOINT_BASELINE).exists())
print(Path(CHECKPOINT_BASELINE).resolve())

In [ ]:
from pathlib import Path

if Path(CHECKPOINT_BASELINE).exists():
    print("Found baseline checkpoint. Loading model and skipping training.")

    ckpt = torch.load(
        CHECKPOINT_BASELINE,
        map_location=device,
        weights_only=False
    )

    baseline_model.load_state_dict(ckpt["model_state_dict"])
    baseline_model.to(device)

    baseline_history = ckpt["history"]

else:
    print("No baseline checkpoint found. Start training from scratch.")

    baseline_history = train_baseline_fcn(baseline_model)

plot_training_history(baseline_history, title_prefix="Baseline FCN")

In [ ]:
_, val_loader = build_voc_loaders(batch_size=1)

val_ds_for_vis = val_loader.dataset

visualize_predictions(
    baseline_model,
    val_ds_for_vis,
    num_samples=10,
    title="Baseline FCN - Validation Samples"
)

In [ ]:
baseline_eval_model = BaselineFCN(backbone_efficientnet).to(device)

baseline_eval_stats = evaluate_checkpoint(
    baseline_eval_model,
    CHECKPOINT_BASELINE,
    split="val"
)

# Task 2: Improve the baseline FCN model (8 marks)

In [ ]:
# ===== Task 2: Residual Multi-scale FPN FCN =====

CHECKPOINT_MULTISCALE = "checkpoints/fcn_multiscale.pth"

MULTISCALE_EPOCHS = 25
MULTISCALE_BATCH_SIZE = 8
MULTISCALE_LR_BASE = 1e-5
MULTISCALE_LR_FPN = 3e-4


class ResidualFPNFCN(nn.Module):
    """
    Multi-scale FCN with residual FPN refinement.

    The model keeps the Task 1 baseline FCN branch and adds a multi-scale FPN branch.
    The final output is:

        baseline_logits + refinement_logits

    This makes the model start from the trained baseline FCN instead of random predictions.
    """

    def __init__(self, backbone, num_classes=NUM_CLASSES, fpn_channels=64):
        super().__init__()

        self.backbone = get_efficientnet_features(backbone)

        # Same layers as Task 1 BaselineFCN.
        # These names match the baseline checkpoint keys: score and upsample.
        self.score = nn.Conv2d(
            1280,
            num_classes,
            kernel_size=1,
            bias=True
        )

        self.upsample = nn.ConvTranspose2d(
            num_classes,
            num_classes,
            kernel_size=64,
            stride=32,
            padding=16,
            bias=False
        )

        # FPN lateral layers from EfficientNet-B0 multi-scale features.
        self.lat32 = nn.Conv2d(1280, fpn_channels, kernel_size=1)
        self.lat16 = nn.Conv2d(112, fpn_channels, kernel_size=1)
        self.lat8 = nn.Conv2d(40, fpn_channels, kernel_size=1)

        self.smooth16 = nn.Sequential(
            nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True)
        )

        self.smooth8 = nn.Sequential(
            nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True)
        )

        self.refine_score = nn.Conv2d(
            fpn_channels,
            num_classes,
            kernel_size=1
        )

        # Important:
        # Start with zero refinement so the model initially behaves like Task 1 baseline.
        nn.init.zeros_(self.refine_score.weight)
        nn.init.zeros_(self.refine_score.bias)

    def forward(self, x):
        input_size = x.shape[-2:]

        out = x
        x8 = None
        x16 = None
        x32 = None

        # EfficientNet-B0 features:
        # i=3 -> 28x28, 40 channels
        # i=5 -> 14x14, 112 channels
        # i=8 -> 7x7, 1280 channels
        for i, layer in enumerate(self.backbone):
            out = layer(out)

            if i == 3:
                x8 = out
            elif i == 5:
                x16 = out
            elif i == 8:
                x32 = out

        # Baseline FCN branch.
        base_logits = self.upsample(F.relu(self.score(x32)))

        # Top-down FPN refinement branch.
        p32 = self.lat32(x32)

        p16 = self.lat16(x16) + F.interpolate(
            p32,
            size=x16.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        p16 = self.smooth16(p16)

        p8 = self.lat8(x8) + F.interpolate(
            p16,
            size=x8.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        p8 = self.smooth8(p8)

        refine_logits = self.refine_score(p8)

        refine_logits = F.interpolate(
            refine_logits,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        return base_logits + refine_logits

In [ ]:
def initialise_multiscale_from_baseline(model, baseline_ckpt_path=CHECKPOINT_BASELINE):
    """
    Initialise the residual FPN model from the trained Task 1 baseline FCN checkpoint.

    The baseline branch weights are loaded.
    The new FPN refinement layers remain randomly initialised.
    """

    ckpt = torch.load(
        baseline_ckpt_path,
        map_location=device,
        weights_only=False
    )

    missing, unexpected = model.load_state_dict(
        ckpt["model_state_dict"],
        strict=False
    )

    print("Loaded Task 1 baseline weights into Task 2 model.")
    print("Missing keys are expected for new FPN layers:")
    print(missing[:10])
    print("Unexpected keys:")
    print(unexpected[:10])

    return model

In [ ]:
def train_multiscale_fcn(model):
    os.makedirs("checkpoints", exist_ok=True)

    train_loader, val_loader = build_voc_loaders(batch_size=MULTISCALE_BATCH_SIZE)

    # Reduce background dominance.
    # VOC has many background pixels, so foreground IoU can be very low
    # even when pixel accuracy looks acceptable.
    class_weights = torch.ones(NUM_CLASSES, device=device)
    class_weights[0] = 0.3

    criterion = nn.CrossEntropyLoss(
        ignore_index=IGNORE_INDEX,
        weight=class_weights
    )

    # Baseline branch uses smaller LR.
    base_params = (
        list(model.backbone.parameters())
        + list(model.score.parameters())
        + list(model.upsample.parameters())
    )

    # New FPN refinement layers use larger LR.
    fpn_params = (
        list(model.lat32.parameters())
        + list(model.lat16.parameters())
        + list(model.lat8.parameters())
        + list(model.smooth16.parameters())
        + list(model.smooth8.parameters())
        + list(model.refine_score.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": base_params, "lr": MULTISCALE_LR_BASE},
            {"params": fpn_params, "lr": MULTISCALE_LR_FPN},
        ],
        weight_decay=1e-4
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_miou": [],
        "val_miou": [],
    }

    print("Evaluating initial baseline-initialised model before training...")

    init_stats = run_segmentation_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        train=False
    )

    best_miou = init_stats["mean_iou"]
    best_state = copy.deepcopy(model.state_dict())

    print(
        f"Initial val: loss={init_stats['loss']:.4f}, "
        f"acc={init_stats['accuracy']:.4f}, "
        f"mIoU={init_stats['mean_iou']:.4f}"
    )

    for epoch in range(1, MULTISCALE_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{MULTISCALE_EPOCHS}")

        train_stats = run_segmentation_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            train=True
        )

        val_stats = run_segmentation_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
            train=False
        )

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc"].append(train_stats["accuracy"])
        history["val_acc"].append(val_stats["accuracy"])
        history["train_miou"].append(train_stats["mean_iou"])
        history["val_miou"].append(val_stats["mean_iou"])

        print(
            f"train: loss={train_stats['loss']:.4f}, "
            f"acc={train_stats['accuracy']:.4f}, "
            f"mIoU={train_stats['mean_iou']:.4f}"
        )

        print(
            f"val:   loss={val_stats['loss']:.4f}, "
            f"acc={val_stats['accuracy']:.4f}, "
            f"mIoU={val_stats['mean_iou']:.4f}"
        )

        if val_stats["mean_iou"] > best_miou:
            best_miou = val_stats["mean_iou"]
            best_state = copy.deepcopy(model.state_dict())
            print(f"New best validation mIoU: {best_miou:.4f}")

    model.load_state_dict(best_state)

    torch.save(
        {
            "model_state_dict": best_state,
            "best_val_miou": best_miou,
            "history": history,
        },
        CHECKPOINT_MULTISCALE
    )

    print(f"\nSaved checkpoint: {CHECKPOINT_MULTISCALE}")
    print(f"Best validation mIoU: {best_miou:.4f}")

    return history

In [ ]:
multiscale_model = ResidualFPNFCN(
    load_efficientnet_b0_backbone(),
    num_classes=NUM_CLASSES,
    fpn_channels=64
).to(device)

multiscale_model = initialise_multiscale_from_baseline(
    multiscale_model,
    CHECKPOINT_BASELINE
)

print(summary(multiscale_model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), device=device))
print("Multi-scale FCN params:", sum(p.numel() for p in multiscale_model.parameters()))

from pathlib import Path

if Path(CHECKPOINT_MULTISCALE).exists():
    print("Found checkpoint, loading multiscale model and skipping training.")

    ckpt = torch.load(
        CHECKPOINT_MULTISCALE,
        map_location=device,
        weights_only=False
    )

    multiscale_model.load_state_dict(ckpt["model_state_dict"])
    multiscale_model.to(device)

    ms_history = ckpt["history"]

else:
    print("No multiscale checkpoint found. Start training from baseline-initialised residual FPN.")

    ms_history = train_multiscale_fcn(multiscale_model)

plot_training_history(ms_history, title_prefix="Residual FPN FCN")

In [ ]:
multiscale_eval_model = ResidualFPNFCN(
    load_efficientnet_b0_backbone(),
    num_classes=NUM_CLASSES,
    fpn_channels=64
).to(device)

multiscale_eval_stats = evaluate_checkpoint(
    multiscale_eval_model,
    CHECKPOINT_MULTISCALE,
    split="val"
)